In [1]:
import torch
from torch import nn
from torch.nn import functional as F

In [6]:
import torch
from torch import nn
import torch.nn.functional as F

class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.hidden = nn.Linear(20, 256)  
        self.out = nn.Linear(256,10)

    def forward(self, X):
        return self.out(F.relu(self.hidden(X)))  

In [7]:
X = torch.randn(2,20)

net = MLP()

net(X)


tensor([[ 0.4386,  0.4004, -0.0316, -0.0193, -0.0186,  0.3047,  0.3155, -0.4525,
         -0.0431,  0.0400],
        [-0.0495,  0.5055, -0.4416,  0.1947, -0.4185, -0.2109, -0.2380, -0.7216,
          0.2980, -0.5768]], grad_fn=<AddmmBackward0>)

In [8]:
## 自定义MySequential实现顺序存储、顺序调用
class MySequential(nn.Module):
    def __init__(self, *args):
        super().__init__()
        for idx,module in enumerate(args):
            self._modules[str(idx)] = module

    def forward(self,X):
        for block in self._modules.values():
           X = block(X)
        return X

In [9]:
net = MySequential(nn.Linear(20, 256), nn.ReLU(), nn.Linear(256, 10))
net(X)

tensor([[ 0.0983, -0.2098, -0.1150, -0.0993, -0.2286,  0.0527, -0.1658, -0.0456,
          0.0737, -0.1160],
        [ 0.3597,  0.4426, -0.2317, -0.3027,  0.0948,  0.4263, -0.1515, -0.2220,
          0.0158, -0.0888]], grad_fn=<AddmmBackward0>)

In [11]:
## 更加灵活的自定义block
class FixedHiddenMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.rand_weight = torch.rand((20,20),requires_grad=False)
        self.linear = nn.Linear(20,20)
    def forward(self,X):
        X = self.linear(X)
        X = F.relu(torch.mm(X,self.rand_weight) + 1)
        X = self.linear(X)

        ## control stream
        while X.abs().sum() > 1:
            X /= 2
        return X.sum()

net = FixedHiddenMLP()
net(X)

tensor(-0.2302, grad_fn=<SumBackward0>)

In [13]:
class NestMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(20,64),
            nn.ReLU(),
            nn.Linear(64,32),
            nn.ReLU()
        )
        self.linear = nn.Linear(32,16)
    def forward(self,X):
        return self.linear(self.net(X))

In [ ]:
## block可以嵌套block
chimera = nn.Sequential(NestMLP(), nn.Linear(16, 20), FixedHiddenMLP())
chimera(X)

tensor(-0.0587, grad_fn=<SumBackward0>)